In [17]:
# =============================================================================
# 🗑️ STEP 1: RESET DATABASE (Clear Previous Data)
# =============================================================================

import os

home = os.path.expanduser("~")
repo_dir = os.path. join(home, "Projects", "scrape_chinese_social_media")
os.chdir(repo_dir)

print("🗑️ Clearing previous data...")

files_to_delete = ['data.db', 'data.xlsx']
for f in files_to_delete:
    if os.path.exists(f):
        os.remove(f)
        print(f"   ✅ Deleted:  {f}")
    else:
        print(f"   ⚠️ Not found: {f}")

print("\n✅ Database reset!  Ready for Douyin scraping.")

🗑️ Clearing previous data...
   ✅ Deleted:  data.db
   ✅ Deleted:  data.xlsx

✅ Database reset!  Ready for Douyin scraping.


In [8]:
# =============================================================================
# 📝 STEP 2: ADD DOUYIN URLs
# =============================================================================

import os

home = os.path.expanduser("~")
repo_dir = os.path. join(home, "Projects", "scrape_chinese_social_media")
os.chdir(repo_dir)

# =============================================================================
# ✏️ PASTE YOUR DOUYIN URLs HERE (one per line)
# =============================================================================

douyin_urls = """
https://www.iesdouyin.com/share/video/6862533859125251340/?region=CN&mid=6862534011185892109&u_code=0&did=MS4wLjABAAAA_oswJqZmU3zCGUyu8OVAC1UU2mCfmc4viCaQE_FNvdMR9u0Icg6C4KNfO5gmMhFF&iid=MS4wLjABAAAA0wvEGyff1ADjiPCblRQIWtvkG7eNV8LYyZ5_BPkYmSd4qCnN--z9LfPLty-TjygW&with_sec_did=1&titleType=title&share_sign=n02R2vZ5QCppS8kyhh0M5mtEAoEqelWcGWdKq65lS64-&share_version=110900&ts=1716365321&from_aid=2955&from_ssr=1
https://www.iesdouyin.com/share/video/7124980639866064136/?region=CN&mid=7124980721214688008&u_code=0&did=MS4wLjABAAAAFs6wBLkYsgEPdS0toRIdQ4FNZVrRttrZ-gldZ-56LSK3eaHcPqyi3HGlBt4b5WY5&iid=MS4wLjABAAAAcvmrmllesJPQC9ZqkE5KrO3Pltag8SfeuErDYO_x-KbAD5NpQEh0Owx11QG1GDhO&with_sec_did=1&titleType=title&share_sign=F7KrmxLyM3IblwEJJ_M6lglp5BE8bB9DnZKXa92CDkU-&share_version=110900&ts=1716365931&from_aid=2955&from_ssr=1"""

# =============================================================================
# Clean and save URLs
urls = [url.strip() for url in douyin_urls.strip().split('\n') if url.strip()]

if urls:
    with open('urls.txt', 'w', encoding='utf-8') as f:
        f. write('\n'.join(urls))
    
    print(f"✅ Saved {len(urls)} Douyin URLs to urls.txt:\n")
    for i, url in enumerate(urls, 1):
        print(f"   {i}.  {url[: 60]}...")
else:
    print("⚠️ No URLs provided!")
    print("   Edit the 'douyin_urls' variable above and run again.")

✅ Saved 2 Douyin URLs to urls.txt:

   1.  https://www.iesdouyin.com/share/video/6862533859125251340/?r...
   2.  https://www.iesdouyin.com/share/video/7124980639866064136/?r...


In [12]:
# =============================================================================
# 🔧 STEP 3: UPDATE DOUYIN SCRAPER (More Comments + Saved Login)
# =============================================================================

import os

home = os.path.expanduser("~")
repo_dir = os.path. join(home, "Projects", "scrape_chinese_social_media")

# =============================================================================
# ⚙️ SETTINGS
# =============================================================================

MAX_COMMENTS = 500      # Maximum comments to extract
SCROLL_COUNT = 30       # Number of scrolls to load comments
HEADLESS = False        # Show browser (set True after first login)

# =============================================================================

douyin_content = '''import asyncio
from datetime import datetime
from playwright.async_api import async_playwright
import json
import os
import utils
import config
from logging_config import get_logger

logger = get_logger()

# Path to save login session
SESSION_FILE = "douyin_session.json"

async def extract_details_new(page):
    details = {
        "title": None,
        "content": None,
        "like_count": None,
        "comment_count": None,
        "share_count": None,
        "publish_time": None
    }

    try:
        title = await page.locator(
            'xpath=(//div[@data-e2e="user-info"]/div[2]/a/div)[2]'
        ).inner_text()
        details["title"] = title. split("\\n")[0]
        logger.info(f"Title: {details['title']}")
    except Exception as e: 
        logger.warning(f"Title error: {e}")

    try:
        details["content"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[1]/div/h1').inner_text()
        logger.info(f"Content:  {details['content'][: 50]}...")
    except Exception as e: 
        logger.warning(f"Content error:  {e}")

    try:
        details["like_count"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[2]/div[1]/div[1]/span').inner_text()
        logger.info(f"Likes: {details['like_count']}")
    except Exception as e:
        logger.warning(f"Like count error: {e}")

    try: 
        details["comment_count"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[2]/div/div[2]/span').inner_text()
        logger.info(f"Comments:  {details['comment_count']}")
    except Exception as e: 
        logger.warning(f"Comment count error: {e}")

    try: 
        details["share_count"] = await page.locator('xpath=//div[@data-e2e="detail-video-info"]/div[2]/div/div[4]/span').inner_text()
        logger.info(f"Shares: {details['share_count']}")
    except Exception as e:
        logger.warning(f"Share count error:  {e}")

    try:
        publish_time = await page.locator('span[data-e2e="detail-video-publish-time"]').inner_text()
        publish_time = publish_time.replace('发布时间：', '').strip()
        dt_object = datetime.strptime(publish_time. strip(), '%Y-%m-%d %H:%M')
        details["publish_time"] = dt_object.strftime('%Y-%m-%d %H:%M:%S')
        logger.info(f"Publish time: {details['publish_time']}")
    except Exception as e:
        logger.warning(f"Publish time error: {e}")

    return details


async def extract_comments(page, max_comments=''' + str(MAX_COMMENTS) + '''):
    logger.info(f"Extracting comments (max: {max_comments})...")
    comments = []
    
    try:
        # Scroll to load comments
        logger.info("Scrolling to load comments...")
        previous_count = 0
        no_change_count = 0
        
        for i in range(''' + str(SCROLL_COUNT) + '''):
            try:
                await page.evaluate("window.scrollBy(0, 800)")
                await page.wait_for_timeout(2000)
                
                # Count loaded comments
                comment_elems = await page. locator('[data-e2e="comment-item"]').all()
                current_count = len(comment_elems) if comment_elems else 0
                
                if i % 5 == 0:
                    logger.info(f"Scroll {i+1}/''' + str(SCROLL_COUNT) + ''': {current_count} comments loaded")
                
                # Stop if no new comments
                if current_count == previous_count:
                    no_change_count += 1
                    if no_change_count >= 5:
                        logger.info("No new comments loading.  Stopping.")
                        break
                else:
                    no_change_count = 0
                
                previous_count = current_count
                
                if current_count >= max_comments: 
                    break
                    
            except Exception as e: 
                logger.error(f"Scroll error: {e}")
        
        # Extract comments
        comment_selectors = [
            '[data-e2e="comment-item"]',
            '. comment-item',
            '[class*="comment"]'
        ]
        
        comment_elements = None
        for selector in comment_selectors: 
            try: 
                comment_elements = await page.locator(selector).all()
                if comment_elements and len(comment_elements) > 0:
                    logger.info(f"Found {len(comment_elements)} comments")
                    break
            except: 
                continue
        
        if not comment_elements: 
            logger.warning("No comment elements found")
            return comments
        
        # Extract text from each comment
        for idx, comment_elem in enumerate(comment_elements[: max_comments]):
            try:
                text = await comment_elem.inner_text()
                comments.append(text)
            except: 
                continue
        
        logger.info(f"Extracted {len(comments)} raw comments")
        
    except Exception as e:
        logger. error(f"Comment extraction error: {e}")
    
    return comments


async def scrape_post(url, conn):
    logger.info("=" * 50)
    logger.info("Launching Playwright browser...")
    logger.info(f"Target url: {url}")

    async with async_playwright() as p:
        browser = await p. chromium.launch(
            headless=''' + str(HEADLESS) + ''',
            args=['--start-maximized']
        )
        
        # Check for saved session
        if os.path.exists(SESSION_FILE):
            logger.info("📂 Loading saved Douyin session...")
            context = await browser.new_context(
                storage_state=SESSION_FILE,
                viewport={"width": 1920, "height":  1080},
                ignore_https_errors=True
            )
        else:
            logger.info("🆕 No saved session.  May need to dismiss login popup.")
            context = await browser.new_context(
                viewport={"width": 1920, "height":  1080},
                ignore_https_errors=True
            )
        
        page = await context. new_page()
        
        try:
            await page.goto(url)
            await page.wait_for_timeout(5000)

            # Try to dismiss login popup
            try:
                close_btn = page.locator('xpath=//div[contains(text(), "登录后免费畅享高清视频")]/following-sibling::div[1]')
                if await close_btn. count() > 0:
                    await close_btn.click()
                    logger.info("Dismissed login popup")
                    await page.wait_for_timeout(2000)
            except:
                pass
            
            # Save session for future use
            if not os.path. exists(SESSION_FILE):
                try:
                    await context.storage_state(path=SESSION_FILE)
                    logger.info("💾 Session saved for future use")
                except:
                    pass

            details = await extract_details_new(page)
            
            # Extract comments
            comments = await extract_comments(page, max_comments=''' + str(MAX_COMMENTS) + ''')
            comments = utils.extract_douyin_comments_from_text(comments)
            
            comments_json = json.dumps(comments, ensure_ascii=False) if comments else None
            
            item = [{
                'unnamed': None,
                'user_name': details['title']. strip() if details['title'] else None,
                'publication_date': details['publish_time'].strip() if details['publish_time'] else None,
                'content': details['content'].strip() if details['content'] else None,
                'shared_count': utils.chinese_unit_to_number(details['share_count']. strip()) if details['share_count'] else 0,
                'comment_count': utils. chinese_unit_to_number(details['comment_count'].strip()) if details['comment_count'] else 0,
                'like_count': utils.chinese_unit_to_number(details['like_count']. strip()) if details['like_count'] else 0,
                'link1': url,
                'link2': None,
                'content_segmented': None,
                'is_agriculture_related': None,
                'index_number': None,
                'comments':  comments_json
            }]
            
            logger.info(f"Saving:  {details['title'][: 30] if details['title'] else 'Unknown'}...")
            utils.insert_data(conn, config.table_name, item)
            logger.info("✅ Saved!")
            
        except Exception as e: 
            logger.error(f"Error scraping {url}: {e}")
        finally:
            await browser.close()
'''

douyin_file = os.path. join(repo_dir, "scrape_douyin_post.py")

# Backup original
if os.path. exists(douyin_file):
    with open(douyin_file + ".backup", 'w', encoding='utf-8') as f:
        with open(douyin_file, 'r', encoding='utf-8') as orig:
            f. write(orig.read())

# Write new file
with open(douyin_file, 'w', encoding='utf-8') as f:
    f.write(douyin_content)

print("=" * 60)
print("✅ DOUYIN SCRAPER UPDATED!")
print("=" * 60)
print(f"""
⚙️ Settings: 
   - MAX_COMMENTS: {MAX_COMMENTS}
   - SCROLL_COUNT:  {SCROLL_COUNT}
   - HEADLESS: {HEADLESS}

🔧 Features:
   - Saves session (no repeated login popups)
   - Smart scrolling (stops when no new comments)
   - Better error handling

🚀 Run Step 4 to start scraping!
""")

✅ DOUYIN SCRAPER UPDATED!

⚙️ Settings: 
   - MAX_COMMENTS: 500
   - SCROLL_COUNT:  30
   - HEADLESS: False

🔧 Features:
   - Saves session (no repeated login popups)
   - Smart scrolling (stops when no new comments)
   - Better error handling

🚀 Run Step 4 to start scraping!



In [14]:
# =============================================================================
# 🚀 STEP 4: RUN DOUYIN SCRAPER
# =============================================================================

import subprocess
import sys
import os

home = os.path.expanduser("~")
repo_dir = os. path.join(home, "Projects", "scrape_chinese_social_media")
os.chdir(repo_dir)

# Check URLs
with open('urls.txt', 'r', encoding='utf-8') as f:
    urls = [u.strip() for u in f.readlines() if u.strip()]

douyin_urls = [u for u in urls if 'douyin' in u.lower()]

if len(douyin_urls) == 0:
    print("❌ No Douyin URLs found in urls.txt!")
    print("   Please add Douyin URLs first (Step 2).")
else:
    print(f"📋 Found {len(douyin_urls)} Douyin URLs")
    print("=" * 60)
    
    os.environ['PYTHONIOENCODING'] = 'utf-8'
    os.environ['PYTHONUTF8'] = '1'
    
    print("""
╔═════════════════════════════════════════════════════════════╗
║  🎵 DOUYIN SCRAPER                                          ║
╠═════════════════════════════════════════════════════════════╣
║                                                             ║
║  • Browser will open                                        ║
║  • Login popup will be auto-dismissed                       ║
║  • No manual login needed for Douyin!                        ║
║                                                             ║
║  ⚠️ DO NOT close the browser window!                         ║
║                                                             ║
╚═════════════════════════════════════════════════════════════╝
    """)
    
    print("🚀 Starting Douyin scraper...")
    
    result = subprocess.run(
        [sys. executable, 'main.py'],
        env={**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHONUTF8': '1'}
    )
    
    print("=" * 60)
    print("✅ Douyin scraping complete!")

📋 Found 2 Douyin URLs

╔═════════════════════════════════════════════════════════════╗
║  🎵 DOUYIN SCRAPER                                          ║
╠═════════════════════════════════════════════════════════════╣
║                                                             ║
║  • Browser will open                                        ║
║  • Login popup will be auto-dismissed                       ║
║  • No manual login needed for Douyin!                        ║
║                                                             ║
║  ⚠️ DO NOT close the browser window!                         ║
║                                                             ║
╚═════════════════════════════════════════════════════════════╝
    
🚀 Starting Douyin scraper...
✅ Douyin scraping complete!


In [15]:
# =============================================================================
# 📊 VIEW DOUYIN RESULTS (FIXED)
# =============================================================================

import sqlite3
import pandas as pd
import json
import os

home = os.path. expanduser("~")
repo_dir = os.path.  join(home, "Projects", "scrape_chinese_social_media")
os.chdir(repo_dir)

conn = sqlite3.connect('data.db')
df = pd.read_sql_query("SELECT * FROM posts", conn)
conn.close()

print("=" * 60)
print("📊 DOUYIN SCRAPING RESULTS")
print("=" * 60)

print(f"\n📈 Total posts scraped: {len(df)}")

if len(df) > 0:
    # Show data preview
    print(f"\n🔍 Data Preview:")
    display_cols = ['user_name', 'publication_date', 'like_count', 'comment_count', 'shared_count']
    available = [c for c in display_cols if c in df.columns]
    display(df[available])
    
    # Count comments per post
    print(f"\n💬 Comments Comparison:")
    print("-" * 70)
    print(f"{'User':<25} {'Recorded':<12} {'Extracted':<12} {'Status':<15}")
    print("-" * 70)
    
    total_extracted = 0
    
    for i, row in df.  iterrows():
        user = str(row. get('user_name', 'Unknown'))[:24]
        
        # FIXED: Handle different data types for comment_count
        recorded = row.get('comment_count', 0)
        try:
            if pd.isna(recorded):
                recorded = 0
            else:
                recorded = int(float(recorded))
        except:
            recorded = 0
        
        # Count extracted comments
        comments = row.get('comments', '')
        if comments and comments != 'None' and comments != '' and not pd.isna(comments):
            try:
                comment_list = json.loads(comments)
                extracted = len(comment_list)
            except:
                extracted = 0
        else:
            extracted = 0
        
        total_extracted += extracted
        
        # Status
        if extracted > 0:
            status = "✅ Success"
        else:
            status = "❌ No comments"
        
        print(f"{user:<25} {recorded:<12} {extracted:<12} {status:<15}")
    
    print("-" * 70)
    print(f"\n📊 SUMMARY:")
    print(f"   Total posts: {len(df)}")
    print(f"   Total comments extracted: {total_extracted}")
    print(f"   Average per post: {total_extracted / len(df):.0f}")
    
    # Show sample comments
    print(f"\n💬 Sample Comments (First Post):")
    print("-" * 70)
    
    first_comments = df.iloc[0]. get('comments', '')
    if first_comments and first_comments != 'None' and not pd.isna(first_comments):
        try:
            comment_list = json.loads(first_comments)
            for j, comment in enumerate(comment_list[: 5]):  # Show first 5
                if isinstance(comment, dict):
                    username = comment.get('username', 'Unknown')[:15]
                    content = comment.get('content', '')[:50]
                    print(f"   {j+1}. [{username}]: {content}...")
                else:
                    print(f"   {j+1}. {str(comment)[:60]}...")
        except Exception as e:
            print(f"   Error parsing:  {e}")
    
    print("-" * 70)

📊 DOUYIN SCRAPING RESULTS

📈 Total posts scraped: 1

🔍 Data Preview:


,user_name,publication_date,like_count,comment_count,shared_count
0,桥城都匀,2020-08-19 11:36:00,45000.0,5385.0,3518.0



💬 Comments Comparison:
----------------------------------------------------------------------
User                      Recorded     Extracted    Status         
----------------------------------------------------------------------
桥城都匀                      5385         5            ✅ Success      
----------------------------------------------------------------------

📊 SUMMARY:
   Total posts: 1
   Total comments extracted: 5
   Average per post: 5

💬 Sample Comments (First Post):
----------------------------------------------------------------------
   1. [╰☆微笑の孤叶☆╮]: 分工明确...
   2. [用户5360440080368]: 还额外增加了另外两个人的就业...
   3. [太阳当空赵]: 这莫一个简单的动作给三个人创造了就业机会...
   4. [用户7555301617492]: 印度人说了你们不懂我们人口多的烦恼，要充分利用每一个人不能浪费人口...
   5. [刘一闪]: 应该不可能吧...
----------------------------------------------------------------------


In [16]:
# =============================================================================
# 📤 STEP 6: EXPORT TO EXCEL
# =============================================================================

import subprocess
import sys
import os

home = os.path.expanduser("~")
repo_dir = os.path. join(home, "Projects", "scrape_chinese_social_media")
os.chdir(repo_dir)

if not os.path. exists('data.db'):
    print("❌ data. db not found! Run the scraper first.")
else:
    print("📤 Exporting to Excel...")
    
    result = subprocess.run(
        [sys. executable, 'export_excel_data.py'],
        env={**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHONUTF8': '1'}
    )
    
    if os.path.exists('data.xlsx'):
        size_kb = os.path. getsize('data.xlsx') / 1024
        print(f"\n✅ Exported to:  {os.path.join(repo_dir, 'data.xlsx')}")
        print(f"📁 File size: {size_kb:.1f} KB")
        
        # Show preview
        import pandas as pd
        df = pd.read_excel('data.xlsx')
        print(f"\n📊 Excel Preview:")
        display(df)
    else: 
        print("❌ Export failed!")

📤 Exporting to Excel...

✅ Exported to:  C:\Users\wk2lam\Projects\scrape_chinese_social_media\data.xlsx
📁 File size: 6.4 KB

📊 Excel Preview:


,Unnamed: 0,User.name,Publication.date,Content,Share,Comment,Like,Link1,Link2,content_segmented,is_agriculture_related,No.,Comments
0,NaN,桥城都匀,2020-08-19 11:36:00,一起来看看非洲刚果的工人们是如何分工的#农民工 #非洲 #效率,3518,5385,45000,https://www.iesdouyin.com/share/video/68625338...,NaN,NaN,NaN,NaN,"[{""username"": ""╰☆微笑の孤叶☆╮"", ""content"": ""分工明确"", ..."
